In [20]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

print("Setup complete")

Setup complete


In [21]:
content_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{content_path}')
    """
).df()

content_schema

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks before building the rule

My confirmed lane is **content refresh opportunity scoring**.

Before building the baseline rule, I will test two signals:

1. **Staleness** — whether content that has gone longer without an update shows a higher future-decline rate. This signal is connected to FlyRank's refresh flags.

2. **Visibility / volume** — whether content with more past search impressions shows a different future-decline rate. This signal is connected to FlyRank's quick-win prioritization logic.

Each signal will be checked using a bucket table that includes the number of content items (`n`) and the observed future-decline rate.

The `future_decline` proxy is used only to evaluate whether the signals are useful. It will not be used as an input to the final baseline score.

In [22]:
signal_query = f"""
WITH daily_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS feature_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

content_dates AS (
    SELECT
        client_hash_id,
        content_hash_id,
        is_published,
        is_deleted,

        NULLIF(
            GREATEST(
                COALESCE(
                    CASE
                        WHEN last_optimized_date <= DATE '2026-03-21'
                        THEN last_optimized_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_updated_date <= DATE '2026-03-21'
                        THEN content_updated_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_created_date <= DATE '2026-03-21'
                        THEN content_created_date
                    END,
                    DATE '1900-01-01'
                )
            ),
            DATE '1900-01-01'
        ) AS last_known_update_date

    FROM read_parquet('{content_path}')
)

SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.past_impressions,
    d.past_clicks,

    ROUND(
        100.0 * d.past_clicks / NULLIF(d.past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * d.past_sum_position / NULLIF(d.past_impressions, 0),
        4
    ) AS past_avg_position,

    d.feature_days,

    DATE_DIFF(
        'day',
        c.last_known_update_date,
        DATE '2026-03-21'
    ) AS days_since_update,

    CASE
        WHEN
            (1.0 * d.outcome_impressions / d.outcome_days)
            <
            0.80 * (1.0 * d.past_impressions / d.feature_days)
        THEN 1
        ELSE 0
    END AS future_decline

FROM daily_windows d
INNER JOIN content_dates c
    ON d.client_hash_id = c.client_hash_id
   AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_published IS TRUE
    AND COALESCE(c.is_deleted, FALSE) IS FALSE
    AND d.feature_days >= 7
    AND d.outcome_days >= 5
    AND d.past_impressions >= 100
"""

signal_frame = con.sql(signal_query).df()

print("Rows available for signal checks:", len(signal_frame))
print(
    "Future-decline base rate:",
    round(signal_frame["future_decline"].mean(), 4)
)

signal_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available for signal checks: 85967
Future-decline base rate: 0.3249


,client_hash_id,content_hash_id,past_impressions,past_clicks,past_ctr,past_avg_position,feature_days,days_since_update,future_decline
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,645.0,2.0,0.3101,4.5752,21,386,0
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,109.0,0.0,0.0000,3.6514,20,386,1
2,client_73cda7b4e4f265ea,content_05434271b257bb68,894.0,3.0,0.3356,5.5817,21,386,0
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2229.0,16.0,0.7178,3.6402,21,386,1
4,client_73cda7b4e4f265ea,content_2662845f598544ef,122.0,1.0,0.8197,8.3852,21,386,1


In [23]:
con.register("signal_frame_tbl", signal_frame)

staleness_table = con.sql("""
SELECT
    CASE
        WHEN days_since_update IS NULL THEN 'Unknown'
        WHEN days_since_update <= 90 THEN '0–90 days'
        WHEN days_since_update <= 180 THEN '91–180 days'
        WHEN days_since_update <= 365 THEN '181–365 days'
        ELSE '366+ days'
    END AS staleness_bucket,

    COUNT(*) AS n,

    ROUND(
        100.0 * AVG(future_decline),
        2
    ) AS decline_rate_pct,

    ROUND(
        MEDIAN(past_impressions),
        1
    ) AS median_past_impressions

FROM signal_frame_tbl

GROUP BY 1
ORDER BY
    CASE staleness_bucket
        WHEN '0–90 days' THEN 1
        WHEN '91–180 days' THEN 2
        WHEN '181–365 days' THEN 3
        WHEN '366+ days' THEN 4
        ELSE 5
    END
""").df()

staleness_table

,staleness_bucket,n,decline_rate_pct,median_past_impressions
0,0–90 days,39056,35.63,598.0
1,91–180 days,9494,39.26,1289.0
2,181–365 days,28605,25.61,800.0
3,366+ days,8812,33.61,490.5


In [24]:
volume_table = con.sql("""
SELECT
    CASE
        WHEN past_impressions < 500 THEN '100–499'
        WHEN past_impressions < 2000 THEN '500–1,999'
        WHEN past_impressions < 10000 THEN '2,000–9,999'
        ELSE '10,000+'
    END AS impression_bucket,

    COUNT(*) AS n,

    ROUND(
        100.0 * AVG(future_decline),
        2
    ) AS decline_rate_pct,

    ROUND(
        MEDIAN(past_ctr),
        3
    ) AS median_ctr_pct

FROM signal_frame_tbl

GROUP BY 1
ORDER BY
    CASE impression_bucket
        WHEN '100–499' THEN 1
        WHEN '500–1,999' THEN 2
        WHEN '2,000–9,999' THEN 3
        ELSE 4
    END
""").df()

volume_table

,impression_bucket,n,decline_rate_pct,median_ctr_pct
0,100–499,36077,36.22,0.000
1,"500–1,999",29179,31.79,0.178
2,"2,000–9,999",17525,27.03,0.211
3,"10,000+",3186,26.71,0.202


### Signal verdicts

**Staleness verdict: MIXED**

The relationship between staleness and future decline is not monotonic. Pages updated 91–180 days ago had the highest observed decline rate at 39.26%, but pages in the 181–365 day bucket had the lowest rate at 25.61%. Therefore, age alone is not a reliable predictor of decline.

**Visibility/volume verdict: OPPOSITE**

The future-decline rate decreased as past impressions increased. Pages with 100–499 impressions had a 36.22% decline rate, while pages with 10,000 or more impressions had a 26.71% rate. Higher visibility may increase the business impact of a refresh, but it did not indicate a higher probability of decline in this slice.

These results change the baseline design. I will use staleness and visibility as transparent review-priority signals, not claim that they accurately predict future decline.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline rule

A page is prioritized for review when it is at least **91 days since its last known update** and received at least **500 search impressions** during the feature window.

The score combines simple staleness points and visibility points. Staleness represents possible refresh need, while visibility represents the potential business impact of reviewing the page. Visibility is not treated as proof that the page will decline.

Every row receives one reason code and one action label:

- `STALE_VISIBLE` → `REVIEW_FOR_REFRESH`
- `BELOW_REVIEW_THRESHOLD` → `HOLD`

The baseline uses only `days_since_update` and `past_impressions`. It does not use `future_decline` or any future-window value as an input.

In [25]:
import os
import numpy as np
import pandas as pd

baseline = signal_frame.copy()

# Transparent hand-written staleness points
baseline["staleness_points"] = np.select(
    [
        baseline["days_since_update"] >= 366,
        baseline["days_since_update"] >= 181,
        baseline["days_since_update"] >= 91,
    ],
    [50, 35, 20],
    default=0,
)

# Transparent hand-written visibility points
baseline["visibility_points"] = np.select(
    [
        baseline["past_impressions"] >= 10000,
        baseline["past_impressions"] >= 2000,
        baseline["past_impressions"] >= 500,
    ],
    [50, 35, 20],
    default=0,
)

baseline["baseline_score"] = (
    baseline["staleness_points"]
    + baseline["visibility_points"]
)

baseline["selected_for_review"] = (
    (baseline["days_since_update"] >= 91)
    & (baseline["past_impressions"] >= 500)
)

baseline["reason_code"] = np.where(
    baseline["selected_for_review"],
    "STALE_VISIBLE",
    "BELOW_REVIEW_THRESHOLD",
)

baseline["action_label"] = np.where(
    baseline["selected_for_review"],
    "REVIEW_FOR_REFRESH",
    "HOLD",
)

ranked_queue = (
    baseline
    .sort_values(
        by=[
            "baseline_score",
            "past_impressions",
            "days_since_update",
        ],
        ascending=[False, False, False],
        na_position="last",
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "rank",
    range(1, len(ranked_queue) + 1),
)

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "days_since_update",
    "past_impressions",
    "past_clicks",
    "past_ctr",
    "past_avg_position",
    "feature_days",
]

queue_output = ranked_queue[queue_columns]

os.makedirs("work/outputs", exist_ok=True)

csv_path = "work/outputs/baseline_action_score.csv"
queue_output.to_csv(csv_path, index=False)

base_rate = baseline["future_decline"].mean()
precision_at_10 = ranked_queue.head(10)["future_decline"].mean()
precision_at_20 = ranked_queue.head(20)["future_decline"].mean()

print("Rows ranked:", len(ranked_queue))
print("Rows selected for review:", int(baseline["selected_for_review"].sum()))
print("Future-decline base rate:", round(base_rate, 4))
print("Precision@10:", round(precision_at_10, 4))
print("Precision@20:", round(precision_at_20, 4))
print("CSV written to:", csv_path)

queue_output.head(20)

Rows ranked: 85967
Rows selected for review: 28214
Future-decline base rate: 0.3249
Precision@10: 0.2
Precision@20: 0.15
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action_label,days_since_update,past_impressions,past_clicks,past_ctr,past_avg_position,feature_days
0,1,client_e547b89c05043229,content_ec2e0346994fb5a5,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,424,173834.0,1075.0,0.6184,2.6185,19
1,2,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,400,105658.0,282.0,0.2669,3.5264,21
2,3,client_73cda7b4e4f265ea,content_e241d6415ac9e534,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,402,84748.0,214.0,0.2525,3.3775,21
3,4,client_e547b89c05043229,content_963de14b1f58978f,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,457,74463.0,436.0,0.5855,3.9014,19
4,5,client_73cda7b4e4f265ea,content_b17c1d1cb0a346d6,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,386,74306.0,342.0,0.4603,4.7204,21
5,6,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,400,72658.0,278.0,0.3826,5.4793,21
6,7,client_e547b89c05043229,content_f86f77b3ebdc05ee,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,457,71082.0,426.0,0.5993,3.6583,19
7,8,client_73cda7b4e4f265ea,content_cf651123f1085418,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,402,63972.0,105.0,0.1641,6.3635,21
8,9,client_73cda7b4e4f265ea,content_8e1334d6356668e3,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,400,63486.0,1.0,0.0016,4.8476,21
9,10,client_e547b89c05043229,content_77276ad7a26f4905,100,STALE_VISIBLE,REVIEW_FOR_REFRESH,457,55291.0,121.0,0.2188,4.0330,19


### Baseline result

The baseline ranked 85,967 eligible content items and selected 28,214 for review.

The overall future-decline base rate was 32.49%, while precision@10 was 20% and precision@20 was 15%. Therefore, the rule did not outperform the base rate at the top of the queue.

This is consistent with the signal audit: staleness showed a mixed relationship with future decline, while higher visibility showed an opposite relationship. I will keep this baseline unchanged as an honest and transparent comparison point for the Week 5 model.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 manual review

The top-ranked items all received the same maximum baseline score because they were at least 366 days old and had at least 10,000 past impressions.

Because both tested signals were weak for predicting future decline, confidence in these recommendations is low to moderate. The recommendations are intended for human review, not automatic refresh decisions.

In [26]:
top20_review = ranked_queue.head(20).copy()

top20_review["why_it_is_here"] = top20_review.apply(
    lambda row: (
        f"{int(row['days_since_update'])} days since the last known update "
        f"and {int(row['past_impressions']):,} past impressions."
    ),
    axis=1,
)

top20_review["confidence_note"] = (
    "Low-to-moderate: the page has high potential impact, "
    "but the signal audit did not show reliable predictive strength."
)

def wrong_reason(row):
    if row["past_avg_position"] <= 3:
        return (
            "The page already ranks strongly, so it may need only a title or "
            "snippet check rather than a full content refresh."
        )
    elif row["past_ctr"] < 0.10:
        return (
            "The low click rate may be caused by search intent, SERP features, "
            "or broad impressions rather than stale content."
        )
    elif row["past_avg_position"] > 6:
        return (
            "The page may have a relevance or ranking problem that a simple "
            "content refresh would not solve."
        )
    else:
        return (
            "The page may be evergreen, seasonal, or still accurate despite "
            "its age."
        )

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    wrong_reason,
    axis=1,
)

top20_review["observed_future_decline"] = top20_review[
    "future_decline"
]

review_columns = [
    "rank",
    "content_hash_id",
    "action_label",
    "reason_code",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong",
    "observed_future_decline",
]

top20_review[review_columns]

,rank,content_hash_id,action_label,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong,observed_future_decline
0,1,content_ec2e0346994fb5a5,REVIEW_FOR_REFRESH,STALE_VISIBLE,"424 days since the last known update and 173,8...",Low-to-moderate: the page has high potential i...,"The page already ranks strongly, so it may nee...",1
1,2,content_fd2117c2c6790e4b,REVIEW_FOR_REFRESH,STALE_VISIBLE,"400 days since the last known update and 105,6...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",0
2,3,content_e241d6415ac9e534,REVIEW_FOR_REFRESH,STALE_VISIBLE,"402 days since the last known update and 84,74...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",0
3,4,content_963de14b1f58978f,REVIEW_FOR_REFRESH,STALE_VISIBLE,"457 days since the last known update and 74,46...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",1
4,5,content_b17c1d1cb0a346d6,REVIEW_FOR_REFRESH,STALE_VISIBLE,"386 days since the last known update and 74,30...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",0
5,6,content_00d4fdf6e48a2d38,REVIEW_FOR_REFRESH,STALE_VISIBLE,"400 days since the last known update and 72,65...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",0
6,7,content_f86f77b3ebdc05ee,REVIEW_FOR_REFRESH,STALE_VISIBLE,"457 days since the last known update and 71,08...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",0
7,8,content_cf651123f1085418,REVIEW_FOR_REFRESH,STALE_VISIBLE,"402 days since the last known update and 63,97...",Low-to-moderate: the page has high potential i...,The page may have a relevance or ranking probl...,0
8,9,content_8e1334d6356668e3,REVIEW_FOR_REFRESH,STALE_VISIBLE,"400 days since the last known update and 63,48...",Low-to-moderate: the page has high potential i...,The low click rate may be caused by search int...,0
9,10,content_77276ad7a26f4905,REVIEW_FOR_REFRESH,STALE_VISIBLE,"457 days since the last known update and 55,29...",Low-to-moderate: the page has high potential i...,"The page may be evergreen, seasonal, or still ...",0


### Weak picks and leakage check

The rule achieved precision@20 of 15%, below the overall future-decline base rate of 32.49%. Only 3 of the top 20 items showed the later decline proxy, so several high-ranked recommendations were weak picks.

This happened because the rule prioritizes old, high-visibility pages, but the signal checks showed that staleness had a mixed relationship with decline and higher visibility had an opposite relationship.

The baseline score uses only two fields available at the decision moment:

- `days_since_update`
- `past_impressions`

The future label and outcome-window fields are used only after ranking to evaluate the baseline. They are not included in the score, reason code, action label, or exported CSV.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [27]:

weak_picks = (
    top20_review[
        top20_review["future_decline"] == 0
    ]
    .head(5)
    .copy()
)

def explain_weak_pick(row):
    if row["past_ctr"] < 0.10:
        return (
            "Very low CTR may come from search intent or SERP features, "
            "not necessarily stale content."
        )
    elif row["past_avg_position"] <= 3:
        return (
            "The page already ranks strongly, so a full refresh may be "
            "unnecessary."
        )
    elif row["past_avg_position"] > 6:
        return (
            "The page may have a relevance or ranking problem that a "
            "content refresh alone may not fix."
        )
    else:
        return (
            "The page may be evergreen, seasonal, or still accurate "
            "despite its age."
        )

weak_picks["why_this_is_a_weak_pick"] = weak_picks.apply(
    explain_weak_pick,
    axis=1,
)

# Explicit leakage audit
baseline_input_columns = [
    "days_since_update",
    "past_impressions",
]

forbidden_columns = [
    "future_decline",
    "outcome_impressions",
    "outcome_days",
    "selected_for_review",
]

input_leakage = sorted(
    set(baseline_input_columns)
    & set(forbidden_columns)
)

csv_leakage = sorted(
    set(queue_output.columns)
    & {
        "future_decline",
        "outcome_impressions",
        "outcome_days",
    }
)

print("Baseline input columns:", baseline_input_columns)
print("Forbidden inputs found:", input_leakage)
print("Future/outcome fields in exported CSV:", csv_leakage)
print(
    "Leakage check passed:",
    len(input_leakage) == 0 and len(csv_leakage) == 0
)

weak_picks[
    [
        "rank",
        "content_hash_id",
        "days_since_update",
        "past_impressions",
        "past_ctr",
        "past_avg_position",
        "why_this_is_a_weak_pick",
    ]
]

Baseline input columns: ['days_since_update', 'past_impressions']
Forbidden inputs found: []
Future/outcome fields in exported CSV: []
Leakage check passed: True


,rank,content_hash_id,days_since_update,past_impressions,past_ctr,past_avg_position,why_this_is_a_weak_pick
1,2,content_fd2117c2c6790e4b,400,105658.0,0.2669,3.5264,"The page may be evergreen, seasonal, or still ..."
2,3,content_e241d6415ac9e534,402,84748.0,0.2525,3.3775,"The page may be evergreen, seasonal, or still ..."
4,5,content_b17c1d1cb0a346d6,386,74306.0,0.4603,4.7204,"The page may be evergreen, seasonal, or still ..."
5,6,content_00d4fdf6e48a2d38,400,72658.0,0.3826,5.4793,"The page may be evergreen, seasonal, or still ..."
6,7,content_f86f77b3ebdc05ee,457,71082.0,0.5993,3.6583,"The page may be evergreen, seasonal, or still ..."


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.